# 05 COF 性质预测：完成第一个可靠 baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/05_cof_property_prediction.ipynb)

这是主线课程最后一章。目标是把前面学到的数据、feature、target、split 和评价指标串起来。

## 1. 今天要完成什么？
我们用一个教学数据集预测人工构造的 `CO2_uptake_demo`。

完整流程：
`选择 feature → 划分 train/test → 预处理 → 训练模型 → 预测 → 计算误差 → 判断结果是否可信`

**注意：这个 target 是教学用人工数据，不能用于科研结论。**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
url='https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/data/cof_demo.csv'
df=pd.read_csv(url)
display(df.head())
print('shape =',df.shape)

## 2. 选择输入和输出
数值 feature 可以直接作为数字输入；`family`、`functional_group` 这类文字类别需要转换成数值编码。

下面用 `OneHotEncoder` 自动完成类别编码。你暂时不需要掌握它的数学细节，只要知道：模型最终只能接收数值。

In [ ]:
target='CO2_uptake_demo'
features=['family','functional_group','pore_A','void_fraction','density','N_fraction','O_fraction']
X=df[features]; y=df[target]
cat=['family','functional_group']
num=[c for c in features if c not in cat]
pre=ColumnTransformer([
 ('num',StandardScaler(),num),
 ('cat',OneHotEncoder(handle_unknown='ignore'),cat)
])
model=Pipeline([
 ('pre',pre),
 ('rf',RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1))
])
print('features:',features)
print('target  :',target)

## 3. 先做最普通的 random split
random split 会随机抽一部分样本作为 test set。它适合做第一条 baseline，但对 COF 不一定足够严格。

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print('MAE =',mean_absolute_error(y_test,pred))
print('RMSE=',mean_squared_error(y_test,pred)**0.5)
print('R2  =',r2_score(y_test,pred))

## 4. 为什么 COF 还要做 family-aware split？
假设同一个 COF family 中的结构很相似。随机划分时，同一家族可能同时出现在 training 和 test 中。模型测试时其实面对的是“很像训练数据的新样本”。

如果真正目标是预测 **从未见过的 COF family**，应该把整个 family 留在 test set。这个思路叫 **family-aware split**。

In [ ]:
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
train_idx,test_idx=next(splitter.split(X,y,groups=df['family']))
X_train2,X_test2=X.iloc[train_idx],X.iloc[test_idx]
y_train2,y_test2=y.iloc[train_idx],y.iloc[test_idx]
model.fit(X_train2,y_train2)
pred2=model.predict(X_test2)
print('Train families:',sorted(df.iloc[train_idx]['family'].unique()))
print('Test families :',sorted(df.iloc[test_idx]['family'].unique()))
print('MAE =',mean_absolute_error(y_test2,pred2))
print('RMSE=',mean_squared_error(y_test2,pred2)**0.5)
print('R2  =',r2_score(y_test2,pred2))

## 5. 怎样解释两个结果？
如果 random split 很好、family-aware split 明显变差，不代表代码错了。更可能说明：模型擅长在见过的化学空间附近插值，但对新 family 的外推能力有限。

这就是科研中很重要的 **generalization（泛化）** 与 **applicability domain（适用域）** 问题。

## 本章术语表
- baseline：作为起点的简单模型；
- preprocessing：模型训练前的数据处理；
- one-hot encoding：把类别文字转换成数值列；
- random split：随机划分；
- group/family-aware split：按组整体划分；
- generalization：对未见数据的表现；
- applicability domain：模型适用的材料空间。

## Exercises
1. 删除 `functional_group` 后重新训练。
2. 删除 `pore_A` 和 `void_fraction` 后重新训练。
3. 比较 random split 与 family-aware split。
4. 用自己的话解释：为什么“最高 R²”不是材料 ML 的唯一目标？
5. 写一句模型结论，必须同时说明使用了哪一种 split。

### 主线课程完成标准
如果你能独立说清楚 `feature → split → train → predict → error → validation` 这条链路，并知道 COF 中要警惕同家族泄漏，就已经完成 00–05 的核心目标。